# doc-parser — Data Scientist Walkthrough

**doc-parser turns a document (PDF, image, DOCX, XLSX, HTML) into one clean Markdown file for RAG.**

It's a *Docling-first, VLM-fallback* hybrid: Docling parses every page; a per-page quality gate sends only the pages Docling struggles with (scanned / garbled / under-extracted) to a vision model (AWS Bedrock Claude). Most pages stay on Docling.

- **Input:** `.pdf`, `.png/.jpg/.jpeg/.tif/.tiff`, `.docx`, `.xlsx/.xlsm`, `.html/.htm`
- **Output:** a Markdown string (write it to `.md`). Plus per-page routing telemetry and warnings.

This notebook runs end-to-end on the bundled test fixtures. **The HTML / DOCX / XLSX examples need no AWS.** The PDF example demonstrates the gate + graceful fallback (also no AWS), and the scanned/image example calls the VLM (Bedrock).

## 0. Setup

Install deps once from the repo root: `uv sync`. Run this notebook with the project's environment (e.g. `uv run jupyter lab`).

The cell below locates the repo, puts `src/` on the path, and imports the entry point `parse_to_markdown`.

In [ ]:
import os, sys
from pathlib import Path

# Locate the repo root (the directory containing pyproject.toml).
root = Path.cwd()
while root != root.parent and not (root / "pyproject.toml").exists():
    root = root.parent
sys.path.insert(0, str(root / "src"))
os.environ.setdefault("PARSER_LOG_LEVEL", "WARNING")  # quieter

FIX = root / "tests" / "fixtures"
from parser_service.markdown_pipeline import parse_to_markdown

print("repo root:", root)
print("fixtures :", sorted(p.name for p in FIX.glob("*") if p.is_file()))

## 1. Parse HTML → Markdown (no AWS)

HTML (and DOCX/XLSX) take the Docling-only whole-document path — the quality gate is skipped and the VLM is never called.

In [ ]:
res = parse_to_markdown(FIX / "page.html")
print(res["markdown"])
print("\nroutes:", res["page_routes"])
print("warnings:", res["warnings"])

## 2. Parse a digital PDF → Markdown + routing (and a live fallback demo)

`digital_simple.pdf` is a tiny 2-page synthetic PDF. Because it has so little text it *trips the coverage gate*, which makes it a nice demo of the safety net: with **no Bedrock configured**, each page escalates, the VLM call fails fast, and the pipeline **gracefully falls back to Docling's markdown** — you'll see route `vlm-fallback-docling` and a warning per page, but valid markdown still comes out and the run never crashes. With Bedrock set, the route would be `vlm`. (Real text-rich PDFs mostly stay `docling-kept`.)

The result is a dict: `markdown`, `page_routes`, `warnings`.

In [ ]:
res = parse_to_markdown(FIX / "digital_simple.pdf")
print(res["markdown"][:1200], "\n…\n")
for r in res["page_routes"]:
    print(r)
print("warnings:", res["warnings"])

### Understanding `page_routes`
Each page records how it was parsed:
- **`docling-kept`** — Docling was confident; its markdown is used (no VLM, no AWS).
- **`vlm`** — the gate escalated the page; the VLM's markdown replaced Docling's.
- **`vlm-fallback-docling`** — the page was escalated but the VLM returned nothing usable (or wasn't reachable), so Docling's output was kept.

`reason` carries the gate's explanation (e.g. `Layer 1: docling_low_grade=...` or `Layer 2: low_coverage: ...`).

In [ ]:
from collections import Counter
Counter(r["route"] for r in res["page_routes"])

## 3. Office formats (DOCX / XLSX) — whole-doc, no AWS

In [ ]:
for name in ["doc.docx", "sheet.xlsx"]:
    r = parse_to_markdown(FIX / name)
    print(f"===== {name} =====")
    print(r["markdown"][:500], "\n")

## 4. Scanned / image inputs → the VLM (needs Bedrock)

Images and scanned pages are where Docling struggles and the VLM earns its keep. This calls **AWS Bedrock**, so it only runs if credentials + model are configured:

```bash
export AWS_REGION=ap-southeast-2
export BEDROCK_VLM_MODEL=anthropic.claude-3-5-sonnet-20241022-v2:0
```

In [ ]:
if os.environ.get("BEDROCK_VLM_MODEL") and os.environ.get("AWS_REGION"):
    res = parse_to_markdown(FIX / "screenshot.png")
    print(res["markdown"][:800])
    print("\nroutes:", res["page_routes"])
else:
    print("Bedrock env not set — skipping the VLM example.")
    print("Set AWS_REGION + BEDROCK_VLM_MODEL to parse images/scans.")

## 5. A mini batch → one `.md` per document

In production you'd use the CLI:
```bash
uv run python scripts/parse_batch.py --input ./inbox --output ./out --concurrency 4
```
which writes `out/<name>.md` per document plus `route_stats.csv` and `failures.json`. Here's the same idea with the Python API on the no-AWS fixtures:

In [ ]:
import tempfile
out_dir = Path(tempfile.mkdtemp(prefix="docparser_out_"))
for name in ["page.html", "doc.docx", "digital_simple.pdf"]:
    r = parse_to_markdown(FIX / name)
    (out_dir / (Path(name).stem + ".md")).write_text(r["markdown"], encoding="utf-8")
print("wrote:", sorted(p.name for p in out_dir.glob("*.md")), "\u2192", out_dir)

## Recap

- **`parse_to_markdown(path) -> {"markdown", "page_routes", "warnings"}`** is the entry point; the `.md` string is the product.
- **Docling-first, VLM-fallback:** HTML/DOCX/XLSX and text-rich digital PDFs run offline; only escalated pages (scans, garbled, under-extracted, images) call Bedrock.
- **Never raises:** failures land in `warnings` (and `failures.json` in batch mode), so a bad doc won't crash a run.
- **CLI:** `scripts/parse_one.py --format md` (single) and `scripts/parse_batch.py` (folder / S3).
- The JSON schema output is only a benchmarking wrapper (`scripts/run_eval.py`), not the shipped artifact — see the README's Evaluation section.